# Gayatri AI — Model Fine-Tuning

**Base model:** gemma-2-2b-it (fine-tuned for tutoring + agent orchestration)  
**Method:** QLoRA (4-bit LoRA)  
**Output:** `gayatri-Q4_K_M.gguf` (~500MB)  

---

## Instructions

1. Go to **Runtime -> Change runtime type -> GPU (T4)**
2. Click **Runtime -> Run all**
3. Training takes ~2-3 hours on free T4
4. Download `gayatri-Q4_K_M.gguf` from the Files panel


In [ ]:
import os, subprocess, sys

def install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

install(["torch"])
install(["transformers", "peft", "bitsandbytes", "trl", "datasets", "accelerate", "sentencepiece", "protobuf"])
install(["unsloth[colab-new]@git+https://github.com/unslothai/unsloth.git"])
install(["llama-cpp-python"])
install(["huggingface_hub"])

print("\n[OK] Dependencies installed!")

import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB)")
else:
    print("WARNING: No GPU detected. Go to Runtime -> Change runtime type -> GPU (T4)")


In [ ]:
SAVE_DIR = "/content/gayatri_model"
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(f"{SAVE_DIR}/data", exist_ok=True)
print(f"Save directory: {SAVE_DIR}")


In [ ]:
import json, random
random.seed(42)

TUTOR_SYSTEMS = [
    "You are a patient tutor. Guide students to answers through questions. Never give direct answers.",
    "You are a programming tutor. Explain concepts step by step with examples.",
    "You are a math tutor. Break problems into small steps. Check understanding at each step.",
    "You are a science tutor. Use real-world analogies. Make complex ideas simple.",
    "You are an encouraging tutor. Celebrate small wins. Adapt to the student's level.",
]

AGENT_SYSTEM = (
    "You are an AI assistant that orchestrates specialized agents. "
    "When a task needs a specific agent, respond with [AGENT:name] followed by the task description. "
    "Available agents: code_reviewer (review code), code_mentor (teach programming), "
    "practice_generator (create exercises), examiner (assess knowledge), "
    "document_analyzer (read and summarize docs), research_helper (search and summarize). "
    "If no agent is needed, answer directly and helpfully."
)

TUTORING_QA = [
    ("What is a variable?", "Think of a variable as a labeled box where you store information."),
    ("How do I create a variable in Python?", "In Python, you give it a name and a value."),
    ("What is a string?", "A string is text wrapped in quotes."),
    ("What are integers?", "Integers are whole numbers with no decimals. Like 1, 42, -7."),
    ("What is a float?", "A float is a number with a decimal point. Like 3.14 or -0.5."),
    ("What is a boolean?", "A boolean is either True or False - like a light switch."),
    ("What is a list?", "A list is like a shopping list - an ordered collection."),
    ("How do I add to a list?", "Lists have an .append() method."),
    ("What is a for loop?", "A for loop repeats code for each item."),
    ("What is a while loop?", "A while loop runs while a condition is true."),
    ("What is a function?", "A function is reusable code - like a recipe."),
    ("What is a dictionary?", "A dictionary stores key-value pairs."),
    ("What is an if statement?", "An if statement runs code only when a condition is true."),
    ("What is recursion?", "Recursion is when a function calls itself."),
    ("What is OOP?", "OOP organizes code around objects that have data and behaviors."),
]

AGENT_QA = [
    ("Review my code for bugs", "code_reviewer", "Review the following code for bugs, security issues, and best practices."),
    ("Help me understand this concept", "code_mentor", "Explain the concept clearly with examples. Use the Feynman technique."),
    ("Give me practice exercises on Python", "practice_generator", "Create 5 progressive exercises. Start easy, increase difficulty. Include solutions."),
    ("Test my knowledge on this topic", "examiner", "Create a 5-question quiz. Provide feedback after each answer."),
    ("Summarize this document", "document_analyzer", "Read and summarize key points. Extract action items."),
    ("Search for information about", "research_helper", "Search for current information. Provide sources."),
]

train_data = []
val_data = []

for i, (q, a) in enumerate(TUTORING_QA * 30):
    convo = [
        {"role": "system", "content": random.choice(TUTOR_SYSTEMS)},
        {"role": "user", "content": q},
        {"role": "assistant", "content": a},
    ]
    (val_data if i % 5 == 0 else train_data).append({"messages": convo})

for i, (trigger, agent, task) in enumerate(AGENT_QA * 25):
    convo = [
        {"role": "system", "content": AGENT_SYSTEM},
        {"role": "user", "content": trigger},
        {"role": "assistant", "content": f"[AGENT:{agent}]\n{task}"},
    ]
    (val_data if i % 5 == 0 else train_data).append({"messages": convo})

random.shuffle(train_data)
random.shuffle(val_data)

for split, data in [("train", train_data), ("val", val_data)]:
    path = f"{SAVE_DIR}/data/{split}.jsonl"
    with open(path, "w") as f:
        for ex in data:
            f.write(json.dumps(ex) + "\n")

print(f"Training examples: {len(train_data)}")
print(f"Validation examples: {len(val_data)}")


In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "unsloth/gemma-2-2b-it"
MAX_SEQ_LENGTH = 2048

print(f"Loading {MODEL_NAME}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print(f"[OK] Model: {MODEL_NAME}")
print(f"Trainable params: {model.get_nb_trainable_parameters()}")


In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset
import json

tokenizer = get_chat_template(tokenizer, chat_template="gemma")

def format_for_training(examples):
    messages = examples["messages"]
    if isinstance(messages, list) and len(messages) > 0 and isinstance(messages[0], str):
        messages = [json.loads(m) for m in messages]
    texts = []
    for conversation in messages:
        text = tokenizer.apply_chat_template(
            conversation, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

train_path = f"{SAVE_DIR}/data/train.jsonl"
val_path = f"{SAVE_DIR}/data/val.jsonl"

train_dataset = load_dataset("json", data_files=train_path, split="train")
val_dataset = load_dataset("json", data_files=val_path, split="train")

train_dataset = train_dataset.map(format_for_training, batched=True, remove_columns=train_dataset.column_names)
val_dataset = val_dataset.map(format_for_training, batched=True, remove_columns=val_dataset.column_names)

print(f"Training: {len(train_dataset)} | Validation: {len(val_dataset)}")
print(f"Sample: {train_dataset[0]['text'][:400]}")


In [ ]:
from trl import SFTTrainer, SFTConfig
import torch

print("Starting training...")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=SFTConfig(
        output_dir=f"{SAVE_DIR}/lora",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        save_steps=100,
        eval_strategy="steps",
        eval_steps=100,
        warmup_steps=20,
        report_to="none",
        seed=42,
    ),
)

trainer.train()
print("\n[OK] Training complete!")


In [ ]:
print("Testing fine-tuned model...\n")
FastLanguageModel.for_inference(model)

test_cases = [
    ("tutoring", [{"role": "system", "content": "You are a patient tutor."}, {"role": "user", "content": "What is a variable?"}]),
    ("agent dispatch", [{"role": "system", "content": AGENT_SYSTEM}, {"role": "user", "content": "Review my code for bugs"}]),
]

for label, messages in test_cases:
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)
    outputs = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.7, top_p=0.9, do_sample=True)
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"--- {label} ---")
    print(response)
    print()


In [ ]:
lora_path = os.path.join(SAVE_DIR, "lora")
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)
print(f"[OK] LoRA adapter saved: {lora_path}")


In [ ]:
print("Merging LoRA adapter...")
merged_path = os.path.join(SAVE_DIR, "gayatri-merged")
model.save_pretrained_merged(merged_path, tokenizer, save_method="merged_16bit")
print(f"[OK] Merged: {merged_path}")


In [ ]:
import subprocess, glob

convert_script = glob.glob("/content/llama.cpp/convert_hf_to_gguf.py")[0]
gguf_path = os.path.join(SAVE_DIR, "gayatri-base.gguf")

print("Converting to GGUF f16...")
result = subprocess.run(
    ["python", convert_script, merged_path, "--outfile", gguf_path, "--outtype", "f16"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("FAILED:", result.stderr[-500:])
    raise RuntimeError("GGUF conversion failed")

print(f"[OK] Base GGUF: {gguf_path} ({os.path.getsize(gguf_path)/1024/1024:.1f} MB)")


In [ ]:
import subprocess, os

print("Building llama.cpp quantizer (~2 min)...")
r = subprocess.run(["bash", "-c", "cd /content/llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release"], capture_output=True, text=True)
if r.returncode != 0:
    print("cmake error:", r.stderr[-300:])
    raise RuntimeError("cmake failed")

r = subprocess.run(["bash", "-c", "cd /content/llama.cpp && cmake --build build --config Release -j$(nproc)"], capture_output=True, text=True)
if r.returncode != 0:
    print("build error:", r.stderr[-500:])
    raise RuntimeError("llama.cpp build failed")

bin_dir = "/content/llama.cpp/build/bin"
quantize_bin = None
for name in ["quantize", "llama-quantize"]:
    p = os.path.join(bin_dir, name)
    if os.path.isfile(p):
        quantize_bin = p
        break

if quantize_bin is None:
    print("Quantize binary not found, building target...")
    subprocess.run(["bash", "-c", "cd /content/llama.cpp && cmake --build build --target quantize -j$(nproc)"])
    for name in ["quantize", "llama-quantize"]:
        p = os.path.join(bin_dir, name)
        if os.path.isfile(p):
            quantize_bin = p
            break

if quantize_bin is None:
    available = os.listdir(bin_dir) if os.path.isdir(bin_dir) else "no build/bin dir"
    raise FileNotFoundError(f"Quantize binary not in {bin_dir}. Available: {available}")

print(f"Using: {quantize_bin}")

quantized_path = os.path.join(SAVE_DIR, "gayatri-Q4_K_M.gguf")
print(f"Quantizing to Q4_K_M...")
r = subprocess.run([quantize_bin, gguf_path, quantized_path, "Q4_K_M"], capture_output=True, text=True)
if r.returncode != 0:
    print("QUANTIZE FAILED:", r.stderr[-500:])
    raise RuntimeError("Quantization failed")

size_mb = os.path.getsize(quantized_path) / 1024 / 1024
print(f"\n{'='*50}")
print(f"[OK] FINAL GGUF: {quantized_path}")
print(f"   Size: {size_mb:.1f} MB | Format: Q4_K_M")
print(f"{'='*50}")
print("Download from the Colab Files panel (folder icon on the left)")


In [ ]:
from google.colab import files
import os

gguf_file = "/content/gayatri_model/gayatri-Q4_K_M.gguf"
if os.path.exists(gguf_file):
    size_mb = os.path.getsize(gguf_file) / 1024 / 1024
    print(f"Downloading {gguf_file} ({size_mb:.1f} MB)...")
    files.download(gguf_file)
else:
    print(f"File not found: {gguf_file}")
